In [10]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()

  Activating new project at `~/GitHub/spacecraft-attitude-course/mekf`


In [11]:
using LinearAlgebra

In [3]:
#quaternion functions
function hat(v)
    return [0 -v[3] v[2];
            v[3] 0 -v[1];
            -v[2] v[1] 0]
end

function unhat(S)
    return 0.5*[S[3,2]-S[2,3];
                S[1,3]-S[3,1];
                S[2,1]-S[1,2]]
end

H = [zeros(1,3); I];
T = [1  zeros(1,3);
     zeros(3,1) -I];

function L(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I + hat(q[2:4])]
end

function R(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I - hat(q[2:4])]
end

function G(q)
    return L(q)*H
end

function Q(q)
    return H'*(R(q)'*L(q))*H
end

function expq(ϕ)
    θ = norm(ϕ)
    return [cos(θ); ϕ*sinc(θ/π)];
end

function logq(q)
    θ = acos(q[1])
    r = q[2:4]/norm(q[2:4])
    return θ*r
end

logq (generic function with 1 method)

In [4]:
h = 0.05 #time step

0.05

In [5]:
function state_prediction(x,u,h)
    q = x[1:4]
    β = x[5:7]
    ω = u-β
    Δq = expq(0.5*h*ω)
    return [L(q)*Δq; β]
end

function state_prediction_deriv(x,u,h)
    q = x[1:4]
    β = x[5:7]
    ω = u-β
    Δq = expq(0.5*h*ω)
    qn = L(q)*Δq
    A = [G(qn)'*R(Δq)*G(q) -0.5*h*G(qn)'*G(q);
         zeros(3,3)            I(3)]
    return A
end

state_prediction_deriv (generic function with 1 method)

In [6]:
function measurement_prediction(x,r_N)
    q = x[1:4]
    Qk = Q(q)
    y = Qk'*r_N
    return y[:]
end

function measurement_prediction_deriv(x,r_N)
    q = x[1:4]
    m = size(r_N,2)
    C = zeros(3*m,6)
    for k = 1:m
        C[((k-1)*3).+(1:3),1:3] .= H'*(L(q)'*L(H*r_N[:,k]) + R(q)*R(H*r_N[:,k])*T)*G(q)
    end
    return C
end

measurement_prediction_deriv (generic function with 1 method)

In [8]:
# Filter Initialization
x = zeros(7)
x[1:4,1] = [1.0; 0; 0; 0];

P = zeros(6,6)
P[:,:,1] .= 0.5*I(6);

In [12]:
using SerialPorts

In [13]:
serial_port = "/dev/tty.usbmodem101" #fill this in with correct port
baud_rate = 9600

9600

In [ ]:
ser = SerialPort(serial_port, baud_rate)

# setup handler at exit to close the serial port
atexit_handler() = (println("closing serial port..."); close(ser))
atexit(atexit_handler)

In [ ]:
function read_line(port::SerialPort)
    buffer = IOBuffer()
    while true
        byte = read(port, UInt8)
        if byte == UInt8('\n')
            break
        end
        write(buffer, byte)
    end
    return String(take!(buffer))
end

In [ ]:
#test that we can read stuff...
while true
    if bytesavailable(ser) > 0
        line = read_line(ser)
        println(line)
    end
end

In [ ]:
for k = 1:(n-1)
    #Prediction
    xpred = state_prediction(x,gyro,h)
    A = state_prediction_deriv(x,gyro,h)
    Ppred = A*P*A' + V

    #Innovation
    z = y - measurement_prediction(xpred,r_N)
    C = measurement_prediction_deriv(xpred,r_N)
    S = C*Ppred*C' + W

    #Kalman Gain
    K = Ppred*C'/S

    #Update
    Δx = K*z
    ϕ = Δx[1:3]
    Δq = [sqrt(1-ϕ'*ϕ); ϕ]
    x = [L(xpred[1:4])*Δq; xpred[5:7]+Δx[4:6]]
    
    P = (I(6) - K*C)*Ppred*(I(6) - K*C)' + K*W*K'
end